In [ ]:
import re
import pandas as pd

def clean_leading_prefix(citation):
    match = re.match(r'^([לבוה])\s*([א-ת"]+)', citation)
    if not match:
        return citation
    prefix = match.group(1)
    maybe_acronym = match.group(2)
    
    # בונה את הצירוף המלא כולל הקידומת
    full = prefix + maybe_acronym

    # מנרמל (מסיר גרשיים) בשביל להשוות לרשימת ראשי התיבות
    def normalize(text):
        return text.replace('"', '').replace("״", "").replace("'", "").replace("׳", "")

    norm_maybe = normalize(maybe_acronym)
    norm_full = normalize(full)

    # תנאי הסרה: החלק שאחרי הקידומת מוכר, אבל הצירוף כולו לא מוכר
    if norm_maybe in acronyms and norm_full not in acronyms:
        return citation[len(prefix):].lstrip()

    return citation

# List of legal acronyms (same as yours)
# הבעיה היא שהיתה פסיק ב-"עפ\"ג" שזה סוגרת מחרוזת ומתחיל סינטקס שגוי - חייבים לכתוב 'עפ"ג' (פסיקים בעברית חייבים במרכאות יחידנות או להכפיל גרשיים)
acronyms = [
    "אב", "אבע", "אימוצ", "אמצ", "אפ", "אפח", "את", "אתפ", "באפ", "באש", "בבנ", "בגצ", "בדא", "בדמ",
    "בדמש", "בהנ", "בהע", "בהש", "בידמ", "בידע", "בל", "בלמ", "במ", "בעא", "בעח", "בעמ", "בעק", "בפ",
    "בפמ", "בפת", "בצא", "בצהמ", "בק", "בקמ", "בקשה", "ברמ", "ברע", "ברש", "בש", "בשא",
    "בשגצ", "בשהת", "בשז", "בשמ", "בשע", "בשפ", "בתת", "גזז", "גמר", "גפ", "דבע", "דח", "דט", "דיונ",
    "דמ", "דמר", "דמש", "דנ", "דנא", "דנגצ", "דנמ", "דנפ", "הד", "הדפ", "הוצלפ", "הט", "הכ", "המ",
    "המד", "הממ", "המע", "המש", "הנ", "הסת", "הע", "העז", "הפ", "הפב", "הפמ", "הצמ", "הש", "השא",
    "השגצ", "השפ", "השר", "הת", "וחק", "וע", "ושמ", "ושק", "ושר", "זי", "חא", "חבר", "חד", "חדא",
    "חדלפ", "חדלת", "חדמ", "חדפ", "חהע", "חי", "חנ", "חסמ", "חעמ", "חעק", "חש", "יוש", "ייתא", "ימא",
    "יס", "כצ", "מ", "מא", "מבכ", "מבס", "מונופולינ", "מזג", "מח", "מחוז", "מחע", "מט", "מטכל", "מי",
    "מיב", "מכ", "ממ", "מס", "מסט", "מעי", "מעת", "מקמ", "מרכז", "מת", "נ", "נב", "נבא", "נמ", "נמב",
    "נעד", "נער", "סבא", "סע", "סעש", "סק", "סקכ", "ע", "עא", "עאח", "עאפ", "עב", "עבאפ", "עבז", "עבח",
    "עבי", "עבל", "עבמצ", "עבעח", "עבפ", "עבר", "עבשהת", "עגר", "עדי", "עדמ", "עהג", "עהס", "עהפ",
    "עו", "עורפ", "עז", "עח", "עחא", "עחדלפ", "עחדפ", "עחדת", "עחהס", "עחע", "עחק", "עחר", "עכב",
    "על", "עלא", "עלבש", "עלח", "עלע", "עמ", "עמא", "עמה", "עמז", "עמח", "עמי", "עמלע", "עממ", "עמנ",
    "עמפ", "עמצ", "עמק", "עמרמ", "עמש", "עמשמ", "עמת", "ענ", "ענא", "ענמ", "ענמא", "ענמש", "ענפ",
    "עסא", "עסק", "עע", "עעא", "עעמ", "עער", "עעתא", "עפ", "עפא", "עפג", "עפהג", "עפמ", "עפמק",
    "עפנ", "עפס", "עפספ", "עפע", "עפר", "עפת", "עצמ", "עק", "עקג", "עקמ", "עקנ", "עקפ", "ער", "ערא",
    "ערגצ", "ערמ", "ערעור", "ערפ", "ערר", "עש", "עשא", "עשמ", "עשר", "עשת", "עשתש", "עת", "עתא",
    "עתמ", "עתפב", "עתצ", "פא", "פה", "פל", "פלא", "פמ", "פמר", "פעמ", "פקח", "פר", "פרק", "פשז",
    "פשר", "פת", "צא", "צבנ", "צה", "צו", "צח", "צמ", "קג", "קפ", "רחדפ", "רמש", "רע", "רעא", "רעב",
    "רעבס", "רעו", "רעמ", "רעס", "רעפ", "רעפא", "רעצ", "רער", "רערצ", "רעש", "רעתא", "רצפ", "רתק",
    "ש", "שבד", "שמ", "שמי", "שנא", "שע", "שעמ", "שק", "שש", "תא", "תאדמ", "תאח", "תאמ", "תאק", "תב",
    "תבכ", "תבע", "תג", "תגא", "תד", "תדא", "תהג", "תהנ", "תהס", "תוב", "תוח", "תח", "תחפ", "תחת",
    "תט", "תי", "תכ", "תלא", "תלב", "תלהמ", "תלפ", "תלתמ", "תמ", "תמהח", "תממ", "תמק", "תמר",
    "תמש", "תנג", "תנז", "תע", "תעא", "תעז", "תפ", "תפב", "תפח", "תפחע", "תפכ", "תפמ", "תפע",
    "תפק", "תצ", "תק", "תקח", "תקמ", "תרמ", "תת", "תתח", "תתע", "תתעא", "תתק"
]

def create_acronym_variants(acronyms):
    acronym_variants = []
    for a in acronyms:
        if len(a) > 1:
            # Case 1: Original acronym with quotes/dots before last letter
            base_acronym = a
            if a.startswith('ב') or a.startswith('ו') or a.startswith('ה'):
                # Also add variant without the prefix letter
                base_acronym = a[1:]
            
            # For each acronym (both with and without prefix)
            for acr in [a, base_acronym]:
                if len(acr) > 1:
                    # Standard quote/dot before last letter, or plain acronym
                    quoted = rf"{acr[:-1]}[\"'״]{acr[-1]}"
                    with_dot = rf"{acr[:-1]}\.{acr[-1]}"
                    plain = acr  # Also match without quotes
                    acronym_variants.append(f"(?:{quoted}|{with_dot}|{plain})")
                    
                    # Add dot-separated variant
                    dots_between = '\.'.join(list(acr))
                    acronym_variants.append(dots_between)
    
    return '|'.join(acronym_variants)
        
acronym_pattern = create_acronym_variants(acronyms)

# Ensure the numbers follow the correct format
number_pattern = r'''
    (?:
        \d{1,8}[-/]\d{1,4}[-/]\d{1,4}  # Format: 35396-04-22, 548070-01-21, 31067-11-11, 2024-04-21
        | \d{1,8}[-/]\d{1,8}             # Format: 895/09
        | \d{1,8}-\d{1,4}-\d{1,4}        # Format: 31067-11-11 (hyphenated)
    )
'''
citation_pattern = fr'''
    (?<!\w)                      # Ensure no letter before
    ([א-ת]?)                     # Optional single Hebrew prefix letter (but no isolated matches)
    ({acronym_pattern})           # Captures acronym (short & long)
    \.?                          # Optional dot after acronym
    \s*                          # Optional spaces after acronym
    (\((.*?)\))?                  # Optional court location in parentheses
    \s*                          # Optional spaces after parentheses
    ({number_pattern})            # Captures case number formats (must have space or be directly after)
    (?!\w)                       # Ensure no letter after
'''.strip()

# Compile regex with verbose flag for readability
citation_regex = re.compile(citation_pattern, re.VERBOSE)


def extract_citations_from_csv(csv_data):
    citations = []
    text_column = csv_data["text"].astype(str)  # Convert to string to avoid NaN issues
    pd.set_option("display.max_colwidth", None)  # Ensure full text is displayed
    # print("\n".join(text_column))  # Print each row as a full text
    # for i, text in enumerate(text_column, 1):
    #     print(f"{i}. {text}")

    matches = text_column.str.extractall(citation_regex)  # Extract structured matches
    # print("Extracted Matches:")
    # print(matches)

    # print("Extracted DataFrame:", matches)  # Debugging step
    
    for _, row in matches.iterrows():
        # Build the citation string, joining all valid elements
        citation = " ".join(map(str, filter(pd.notna, row))).strip()

        # Clean up extra spaces
        citation = re.sub(r"\s{2,}", " ", citation)


        # # Remove invalid extra words (e.g., "על 12")
        if re.match(r"^על \d+$", citation):  
            print('Skip invalid cases like "על 12')
            continue  # Skip invalid cases like "על 12"

        # Fix duplicated court locations, e.g., "(מחוזי מרכז) מחוזי מרכז" → "(מחוזי מרכז)"
        citation = re.sub(r"\((.*?)\)\s+\1", r"(\1)", citation)
        citation=clean_leading_prefix(citation)
        # Add the cleaned citation to the list
        citations.append(citation)
    
    # Return citations as a list, even if some are empty or missing optional groups
    return citations if citations else []




In [ ]:
# Debug: Test the regex pattern with the specific examples
test_texts = [
    'ת"פ 35396-04-22 מדינת ישראל',
    'עפ"ג 2024-04-21 מ"י נ\' מנדורי',
    'ת"פ 548070-01-21 מ"י נ\' זהר אלבחירי'
]

print("Testing regex pattern with examples:")
print("=" * 60)
for text in test_texts:
    matches = citation_regex.findall(text)
    print(f"\nText: {text}")
    print(f"Matches found: {len(matches)}")
    for i, match in enumerate(matches, 1):
        print(f"  Match {i}: {match}")
    if not matches:
        # Try to find what's wrong
        print(f"  No matches - testing components:")
        # Test acronym pattern alone
        test_acronym = re.search(fr'({acronym_pattern})', text)
        if test_acronym:
            print(f"    Acronym found: {test_acronym.group()}")
        # Test number pattern alone  
        test_number = re.search(fr'({number_pattern})', text)
        if test_number:
            print(f"    Number found: {test_number.group()}")

In [ ]:
import os
import gc
import torch
import pandas as pd
import docx
import re
from transformers import AutoTokenizer, BertTokenizer, BertForSequenceClassification
from pathlib import Path
from openai import OpenAI
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["OPENAI_API_KEY"] = "REPLACED_OPENAI_KEY"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Define required sections and citation patterns
required_parts = [
    "מתחמי ענישה", "אחידות בענישה", "מתחם הענישה", "מתחם ענישה", "דיון",
    "ענישה נהוגה", "הענישה הנוהגת", "ענישה נוהגת", "מתחם העונש", "מתחם עונש",
    "מדיניות הענישה", "והכרעה", "ההרשעה", "מדיניות הענישה הנהוגה"
]


# Load the trained BERT model and tokenizer
model_path ='/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/classifier_relvant_citation_model.pt'
tokenizer_bert = BertTokenizer.from_pretrained('avichr/heBERT')
model_bert = BertForSequenceClassification.from_pretrained('avichr/heBERT', num_labels=2)
model_bert.load_state_dict(torch.load(model_path, map_location=device))
model_bert.to(device)
model_bert.eval()


def split_preserving_structure(text):
    paragraphs = re.split(r'(?<=\d\.)\s', text)  # Split after numbers followed by a period
    return [para.strip() for para in paragraphs if para.strip()]

# def query_gpt(text,citation):
#     """
#     Queries gpt-4.1-mini to extract and segment legal citations.
#     """
#     prompt = f"""
#     להלן הטקסט המשפטי:

#     {text}

#     המשימה שלך היא לחלץ **רק** את החלק של הטקסט הקשור ישירות לציטוט "{citation}".
    
#     **כללי חילוץ:**
#     - **אל תשנה כל ניסוח.** שמור על הניסוח המקורי בדיוק כפי שהוא מופיע במסמך המסופק.
#     - **אל תסכם או תנסח מחדש.**
#     - **החזר רק את החלק הרלוונטי**, לא את כל הטקסט.
    
#     **מקרה 1: ציטוטים מרובים באותה פסקה:**
#     - אם הציטוט "{citation}" מופיע באותה פסקה יחד עם ציטוטים נוספים (למשל: "...כפי שנקבע בע"פ 2247/10... וע"פ 2251/11..."), 
#       **חלץ את כל ההקשר המשותף** שמופיע לפני הציטוטים + **את הציטוט המבוקש בלבד**.
#     - דוגמה: אם הפסקה מתחילה ב"אתחיל ואדגיש..." ומסתיימת ב"כפי שנקבע בע"פ 2247/10... וע"פ 2251/11", 
#       ובקשת את "ע"פ 2247/10", החזר את כל ההקשר מ"אתחיל ואדגיש..." עד "ע"פ 2247/10..." (ללא הציטוט השני).
    
#     **מקרה 2: ציטוטים בפסקאות נפרדות:**
#     - אם כל ציטוט מוסבר בפסקה נפרדת משלו (למשל: "בע"פ 1371/05... [הסבר מלא]. בת.פ. 25802-02-10... [הסבר מלא]"),
#       **חלץ רק את הפסקה הרלוונטית** לציטוט המבוקש, כולל כל ההסבר של התיק.
#     - אל תכלול פסקאות של ציטוטים אחרים.
    
#     **כללי כלליים:**
#     - אם הציטוט מופיע ברשימה לאחר "ראו למשל ..." או ביטוי דומה, כלול את ההסבר הקודם המשותף לכל הציטוטים ברשימה.
#     - אם הציטוט מוסבר בסעיף ספציפי (למשל, "בע"פ 9373/10 ותד נ' מדינת ישראל..."), חלץ את **כל ההסבר** של התיק.
#     - אל תסיר שום הקשר חשוב לגבי פסק הדין.
#     - **אל** תחלץ רק את הציטוט עצמו (כמו "(רע"פ 2718/04)") ללא העיקרון המשפטי שהוא תומך בו.


#     החזר רק את הטקסט המחולץ. אל תכלול תוכן לא רלוונטי או עיצוב.
#     """
#     print(f"🧠 Sending to GPT for extraction...")

#     try:
#         response = client.chat.completions.create(
#             model="gpt-4.1-mini", 
#             messages=[
#                 {"role": "system", "content": "אתה בינה מלאכותית מאומנת לחלץ ולבנות ציטוטים משפטיים."},
#                 {"role": "user", "content": prompt}
#             ]
#         )

#         processed_text = response.choices[0].message.content
#         return processed_text

#     except Exception as e:
#         print(f"🚨 GPT API error: {e}")
#         return [text]  # Return original text in case of failure
def query_gpt(text,citation):
    """
    Queries gpt-4.1-mini to extract and segment legal citations.
    """
    prompt = f"""
    Given the following legal text:

    {text}

    Your task is to extract **only** the part of the text that directly relates to the citation "{citation}".
    
    **Extraction Rules:**
    - **Do not modify any wording.** Keep the original phrasing exactly as it appears in the provided document.
    - **Do not summarize or rephrase.**
    - **Return only the relevant portion**, not the full text.
    - **Handle grouped citations carefully:**
        - If the citation appears in a list following "ראו למשל ..." or similar, include the preceding explanation that applies to all citations.
        - Do not include other citations from the list—return only the text relevant to "{citation}".
    - **Handle case explanations properly:**
        - If the citation is explained in a specific section (e.g., "בע"פ 9373/10 ותד נ' מדינת ישראל..."), extract the **entire explanation** of the case.
        - Do not remove any important context about the court ruling.
    - Do **not** extract only "(רע"פ 2718/04)" without the legal principle it supports.


    Only return the extracted text. Do not include unrelated content or formatting.
    """
    print(f"🧠 Sending to GPT for extraction...")

    try:
        response = client.chat.completions.create(
            model="gpt-4.1", 
            messages=[
                {"role": "system", "content": "You are an AI trained to extract and structure legal citations."},
                {"role": "user", "content": prompt}
            ]
        )

        processed_text = response.choices[0].message.content
        return processed_text

    except Exception as e:
        print(f"🚨 GPT API error: {e}")
        return [text]  # Return original text in case of failure
    
def filter_csv_relevant_parts(csv_data):
    """
    Extracts the first occurrence of a required part in the CSV and all subsequent rows.
    """
    start_index = None

    # Find the first row containing a required part
    for idx, row in csv_data.iterrows():
        if any(req_part in str(row.get("part", "")) for req_part in required_parts):
            start_index = idx
            break

    # If a match is found, return only relevant rows
    if start_index is not None:
        return csv_data.iloc[start_index:]
    else:
        # print("NO required parts in data")
        # print("parts in data:")
        # print(csv_data["part"].unique())
        return pd.DataFrame(columns=csv_data.columns)  # Return an empty DataFrame if no matches found



# Function to find all occurrences of a citation in the document
def find_all_occurrences(doc, citation):
    indices = []
    for i, paragraph in enumerate(doc.paragraphs):
        if citation in paragraph.text:
            indices.append(i)  # Store all occurrences of the citation
    return indices

# Function to get relevant context for each occurrence of the citation
def get_context_paragraphs(doc, index, citation):
    context_text = []

    # Search for the closest non-empty previous paragraph
    prev_index = index - 1
    while prev_index >= 0 and not doc.paragraphs[prev_index].text.strip():
        prev_index -= 1  # Move backwards until finding text

    if prev_index >= 0:
        context_text.append(doc.paragraphs[prev_index].text.strip())

    # Get the current paragraph (must exist, but check if empty)
    curr_text = doc.paragraphs[index].text.strip()
    if curr_text:
        context_text.append(curr_text)
    else:
        print(f"⚠️ Warning: Empty paragraph for citation {citation} at index {index}. Skipping occurrence.")
        return None  # Skip this occurrence if the current paragraph is empty

    # Search for the closest non-empty next paragraph
    next_index = index + 1
    while next_index < len(doc.paragraphs) and not doc.paragraphs[next_index].text.strip():
        next_index += 1  # Move forward until finding text

    if next_index < len(doc.paragraphs):
        context_text.append(doc.paragraphs[next_index].text.strip())

    # Ensure we have at least one non-empty paragraph
    if not context_text:
        print(f"⚠️ Warning: No valid text found for citation {citation} at index {index}. Skipping occurrence.")
        return None

    return "\n".join(context_text).strip()
def normalize_case_name_2(name):
    if pd.isna(name):
        return ""
    name = str(name)
    name = re.sub(r"\(.*?\)", "", name)
    name = re.sub(r"[∕/\\]", "-", name)
    name = re.sub(r"\s+", " ", name)
    name = name.strip().lower().replace(" ", "_")
    return name


# Function to process and tag document paragraphs
def process_and_tag_with_split(docx_path: str, csv_path: str, output_path: str):
    """
    Process a .docx document and its corresponding CSV, find relevant paragraphs with context, 
    extract relevant text using GPT, tag with BERT, and store results.
    """
    doc = docx.Document(docx_path)
    csv_data = pd.read_csv(csv_path)
    filtered_csv_data = filter_csv_relevant_parts(csv_data)
    if filtered_csv_data.empty:
        # print("⚠️ Skipping file — no relevant parts found.")
        return

    citations = extract_citations_from_csv(filtered_csv_data)
    results = []
    if len(citations) > 30:
        print(f"TOO MANY CITATIONS IN CSV Found {len(citations)}")
        print(docx_path)


        return
    print(f"🔍 Found {len(citations)} citations in CSV")

    for citation in citations:
        citation_indices = find_all_occurrences(doc, citation)  # Find all occurrences

        # Collect all contexts where the citation appears
        merged_contexts = []
        for index in citation_indices:
            full_context = get_context_paragraphs(doc, index, citation)
            if full_context:
                merged_contexts.append(full_context)

        # If no valid contexts found, skip this citation
        if not merged_contexts:
            continue  

        # Merge all valid contexts into one, ensuring uniqueness
        final_context = "\n".join(set(merged_contexts)).strip()  # Remove duplicates
        # print(citation)
        # print(final_context)

        # Ask GPT to extract the relevant part
        extracted_text = query_gpt(final_context, citation)

        # Tag the extracted text with BERT
        encoding = tokenizer_bert(extracted_text, truncation=True, padding=True, max_length=128, return_tensors="pt")
        encoding = {key: val.to(device) for key, val in encoding.items()}
        with torch.no_grad():
            output = model_bert(**encoding)
            prediction = torch.argmax(output.logits, dim=-1).item()


        citation=normalize_case_name_2(citation)    
        # Store only one result per citation
        result = {
            'citation': citation,
            'context_text': final_context,
            'extracted_text': extracted_text,
            'predicted_label': prediction
        }
        results.append(result)

    # Save to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Processed document saved to: {output_path}")

if __name__ == "__main__":

    docx_directory = Path('/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/drugs_docx')
    csv_directory = Path('/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/verdict_csv')
    output_directory = Path('/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/verdicts_tagged_citations')
    output_directory.mkdir(parents=True, exist_ok=True)

    # Stats counters
    total_files = 0
    processed_files = 0
    skipped_empty_or_missing = 0
    missing_csv = 0
    files_with_citations = 0
    total_citations = 0
    total_tagged_as_1 = 0

    all_files = list(docx_directory.glob("*.docx"))
    print(f"🗂 Total DOCX files found: {len(all_files)}")

    for file_path in tqdm(all_files, desc="Processing DOCX files"):
        total_files += 1
        new_file_path = file_path.stem
        csv_file = csv_directory / f"{new_file_path}.csv"
        output_file = output_directory / f"{file_path.stem}.csv"

        if not csv_file.exists():
            print(f"CSV file not found for: {csv_file}")
            missing_csv += 1
            continue

        if output_file.exists() and output_file.stat().st_size > 0:
            try:
                df_existing = pd.read_csv(output_file)
                num_citations = len(df_existing)
                num_tagged_1 = (df_existing["predicted_label"] == 1).sum()

                total_citations += num_citations
                total_tagged_as_1 += num_tagged_1
                files_with_citations += 1
                continue
            except Exception as e:
                print(f"⚠️ Error reading {output_file.name}: {e}")
                skipped_empty_or_missing += 1
                continue

        if output_file.exists() and output_file.stat().st_size == 0:
            skipped_empty_or_missing += 1
            continue

        if not output_file.exists():
            try:
                process_and_tag_with_split(str(file_path), str(csv_file), str(output_file))
            except Exception as e:
                print(f"⚠️ Error processing {file_path.name}: {e}")
                skipped_empty_or_missing += 1
                continue
            if output_file.exists() and output_file.stat().st_size > 0:
                try:
                    df_new = pd.read_csv(output_file)
                    num_citations = len(df_new)
                    num_tagged_1 = (df_new["predicted_label"] == 1).sum()
                    total_citations += num_citations
                    total_tagged_as_1 += num_tagged_1
                    files_with_citations += 1
                    processed_files += 1
                except Exception as e:
                    print(f"⚠️ Failed to read newly written output: {output_file.name}")
                    skipped_empty_or_missing += 1

    # Averages
    avg_citations_per_file = total_citations / files_with_citations if files_with_citations else 0
    avg_tagged_1_per_file = total_tagged_as_1 / files_with_citations if files_with_citations else 0

    print("\n===== 📊 Processing Summary =====")
    print(f"Total DOCX files:               {total_files}")
    print(f"Processed files:                {files_with_citations}  # output file exists and is not empty")
    print(f"Skipped (already processed):    {total_files - files_with_citations}  # output missing or empty")
    print(f"Missing CSV files:              {missing_csv}")
    print(f"Files with citation data:       {files_with_citations}")
    print(f"Total citations:                {total_citations}")
    print(f"Total tagged as 1:              {total_tagged_as_1}")
    print(f"Avg citations per file:         {avg_citations_per_file:.2f}")
    print(f"Avg tagged=1 per file:          {avg_tagged_1_per_file:.2f}")


In [ ]:
# Create a CSV with 200 random examples for manual testing
import random

# Define the output directory (same as in the main processing)
output_directory = Path('/Users/liorb/Library/CloudStorage/OneDrive-post.bgu.ac.il/Thesis!!!/new_try/drugs/verdicts_tagged_citations')

# Collect all CSV files
all_csv_files = list(output_directory.glob("*.csv"))
print(f"📁 Found {len(all_csv_files)} CSV files in output directory")

# Read and combine all CSV files
all_dataframes = []
for csv_file in all_csv_files:
    try:
        df = pd.read_csv(csv_file)
        if not df.empty:
            # Add source file column for reference
            df['source_file'] = csv_file.stem
            all_dataframes.append(df)
    except Exception as e:
        print(f"⚠️ Error reading {csv_file.name}: {e}")
        continue

if not all_dataframes:
    print("❌ No data found in any CSV files!")
else:
    # Combine all dataframes
    combined_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"📊 Total citations found: {len(combined_df)}")
    
    # Sample 200 random rows (or all if less than 200)
    sample_size = min(200, len(combined_df))
    random_sample = combined_df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    # Save to CSV
    output_path = output_directory.parent / "random_200_examples_for_testing.csv"
    random_sample.to_csv(output_path, index=False, encoding="utf-8")
    print(f"✅ Saved {sample_size} random examples to: {output_path}")
    print(f"📋 Columns in output: {list(random_sample.columns)}")
    print(f"📈 Label distribution:")
    print(random_sample['predicted_label'].value_counts())